# CO-SPY pretrained — Eval nhanh FLUX.1-schnell (MSCOCO)

Chỉ eval **5.000 real MSCOCO val2017 + 5.000 fake FLUX.1-schnell** bằng **CO-SPY pretrained Fusion v1**.

- Không chạy toàn Co-SpyBench.
- Có progress bar.
- Xuất AP, Accuracy và CSV dự đoán.
- Bật **GPU** + **Internet** trên Kaggle.

In [ ]:
from pathlib import Path
import os, sys, json, shutil, subprocess, time

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
REPO = WORK / "cospy"

BATCH_SIZE = 32
NUM_WORKERS = 4
N_REAL = 5000
N_FAKE = 5000
SEED = 1024

print("INPUT:", INPUT)

In [ ]:
# Clone source + dependencies
if not REPO.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/datnguyentrung/CO-SPY.git", str(REPO)],
        check=True,
    )

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "open-clip-torch==2.31.0",
    "diffusers==0.32.2",
    "transformers==4.46.3",
    "accelerate==1.0.1",
    "safetensors==0.5.3",
    "ftfy==6.2.3",
    "timm==1.0.15",
    "loguru==0.7.3",
], check=True)

print("Repo ready:", REPO)

In [ ]:
# Auto-discover pretrained + real/fake inputs
IMG_EXTS = {".jpg", ".jpeg", ".png", ".webp"}

def find_weight(name):
    xs = list(INPUT.rglob(name))
    if not xs:
        raise FileNotFoundError(name)
    xs.sort(key=lambda p: ("__MACOSX" in str(p), len(str(p))))
    return xs[0]

weights = {
    "artifact_weights.pth": find_weight("artifact_weights.pth"),
    "semantic_weights.pth": find_weight("semantic_weights.pth"),
    "fusion_weights.pth": find_weight("fusion_weights.pth"),
}

dst_dir = REPO / "pretrained" / "sd-v1_4"
dst_dir.mkdir(parents=True, exist_ok=True)

for name, src in weights.items():
    dst = dst_dir / name
    if dst.exists() or dst.is_symlink():
        dst.unlink()
    dst.symlink_to(src)
    print("[WEIGHT]", name, "->", src)

def direct_image_count(p):
    try:
        return sum(1 for f in p.iterdir() if f.is_file() and f.suffix.lower() in IMG_EXTS)
    except:
        return 0

# Real = MSCOCO val2017, usually exactly 5000
real_candidates = []
for p in INPUT.rglob("val2017"):
    if p.is_dir():
        n = direct_image_count(p)
        if n >= 4900:
            real_candidates.append((abs(n-5000), n, p))

if not real_candidates:
    raise RuntimeError("Không tìm thấy MSCOCO val2017 ~5000 ảnh.")

real_candidates.sort(key=lambda x: (x[0], len(str(x[2]))))
REAL_DIR = real_candidates[0][2]
real_images = sorted(
    p for p in REAL_DIR.iterdir()
    if p.is_file() and p.suffix.lower() in IMG_EXTS
)[:N_REAL]

# Fake = any input image whose path contains both flux and schnell
fake_images = sorted([
    p for p in INPUT.rglob("*")
    if p.is_file()
    and p.suffix.lower() in IMG_EXTS
    and "flux" in str(p).lower()
    and "schnell" in str(p).lower()
])[:N_FAKE]

print("\nREAL_DIR:", REAL_DIR)
print("Real:", len(real_images))
print("Fake:", len(fake_images))
if fake_images:
    print("Fake example:", fake_images[0])

assert len(real_images) == N_REAL, f"Real {len(real_images)}/{N_REAL}"
assert len(fake_images) == N_FAKE, f"Fake {len(fake_images)}/{N_FAKE}"

print("\n[PASS] 5000 real + 5000 fake ready.")

In [ ]:
# Load pretrained CO-SPY Fusion v1
import torch

os.chdir(REPO)
sys.path.insert(0, str(REPO))

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True

from evaluate import Detector

device = "cuda:0" if torch.cuda.is_available() else "cpu"
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

t0 = time.time()
detector = Detector(
    device=device,
    mode="fusion",
    train_dataset="sd-v1_4",
    pretrain=True,
    ckpt=str(WORK / "eval_ckpt"),
    batch_size=BATCH_SIZE,
    fusion_head="v1",
)
model = detector.model
model.eval()

print(f"Model loaded in {(time.time()-t0)/60:.2f} min")

In [ ]:
# Eval 10k images
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from sklearn.metrics import average_precision_score, accuracy_score

class EvalDataset(Dataset):
    def __init__(self, paths, labels, transform):
        self.paths = paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        with Image.open(self.paths[i]) as im:
            x = self.transform(im.convert("RGB"))
        return x, self.labels[i]

paths = real_images + fake_images
labels = [0]*len(real_images) + [1]*len(fake_images)

ds = EvalDataset(paths, labels, model.test_transform)
loader = DataLoader(
    ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2,
)

y_pred, y_true = [], []
start = time.time()

with torch.inference_mode():
    for images, yy in tqdm(loader, desc="CO-SPY Fusion eval", unit="batch"):
        pred = model.predict(images)
        y_pred.extend(float(v) for v in pred)
        y_true.extend(yy.tolist())

elapsed = time.time() - start
y_pred = np.asarray(y_pred, dtype=np.float64)
y_true = np.asarray(y_true, dtype=np.int64)

ap = average_precision_score(y_true, y_pred)
acc = accuracy_score(y_true, (y_pred >= 0.5).astype(np.int64))
real_ok = (y_pred[y_true == 0] < 0.5).mean()
fake_ok = (y_pred[y_true == 1] >= 0.5).mean()

result = {
    "model": "CO-SPY pretrained Fusion v1",
    "scope": "MSCOCO val2017 5k real vs FLUX.1-schnell 5k fake",
    "AP": float(ap),
    "AP_percent": float(ap*100),
    "Accuracy": float(acc),
    "Accuracy_percent": float(acc*100),
    "Real_correct_percent": float(real_ok*100),
    "Fake_detected_percent": float(fake_ok*100),
    "elapsed_minutes": float(elapsed/60),
    "images_per_second": float(len(y_true)/elapsed),
    "threshold": 0.5,
}

print("\n" + "="*64)
print(f"AP                 : {ap*100:.2f}%")
print(f"Accuracy           : {acc*100:.2f}%")
print(f"Real correctly real: {real_ok*100:.2f}%")
print(f"Fake detected AI   : {fake_ok*100:.2f}%")
print(f"Runtime            : {elapsed/60:.1f} min")
print(f"Throughput         : {len(y_true)/elapsed:.2f} img/s")
print("="*64)
print("Paper full FLUX.1-schnell reference: AP 95.52 | Acc 85.24")
print("Lưu ý: notebook này chỉ eval subset MSCOCO 5k+5k.")

In [ ]:
# Save results
import csv

OUT = WORK / "cospy_flux_schnell_eval"
OUT.mkdir(parents=True, exist_ok=True)

with open(OUT/"result.json", "w", encoding="utf-8") as f:
    json.dump(result, f, indent=2, ensure_ascii=False)

with open(OUT/"predictions.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["path", "label", "score", "predicted_label"])
    for p, y, s in zip(paths, y_true.tolist(), y_pred.tolist()):
        w.writerow([str(p), y, s, int(s >= 0.5)])

print("Saved:")
print(OUT/"result.json")
print(OUT/"predictions.csv")